<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

# Plotly Express Walkthrough

### CP101: Introduction to Urban Data Analytics

**Week 9, Monday** &nbsp;·&nbsp; Data Visualization in Action

**Estimated time:** 20-25 minutes &nbsp;·&nbsp; **Not graded, not submitted.**

<hr style="border: 1px solid #fdb515;" />

### How to work

- Run each code cell with `Shift + Enter`, in order, from the top.
- Every chart is interactive.
  Hover over marks, drag to zoom, double-click to reset, and click legend entries to hide them.

<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

## Learning Objectives

By the end of this walkthrough, you will be able to:

1. Build a chart with one Plotly Express call by naming which DataFrame column sets the x-axis, the y-axis and the color
2. Build a dot plot, a Sankey diagram, a 24-hour polar "clock" and an animated line chart, and explain why each one fits its question
3. Animate a chart with `animation_frame`, keep the frames comparable with a fixed range and a reference line, and say when a static chart reads better

<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

## The Plotly Express pattern

Three of the four charts in this notebook are one function call with the same shape:

```python
px.chart_type(
    data_frame,
    x="column",
    y="column",
    color="column"
    )
```

A table of 625 bike share stations is only numbers until you decide how each row should look on the screen.
That decision has two parts.
First, pick a **mark**, the shape that stands for one row, such as a dot, a line or a bar.
Plotly calls its dots *markers*.
Then pick the **channels**, the properties of that mark that carry the data, such as its position, length, color, size or shape.
In Example 1, each station becomes a dot, its position along the x-axis shows how many trips start there, and its color and shape say whether that is a weekday or a weekend.
In code, each channel is one argument, so `x="avg_trips"` sets the position and `color="day_type"` sets the color.

| Argument | Maps the column to |
|---|---|
| `x`, `y` | Position on the horizontal and vertical axes |
| `color` | Mark color, with a legend |
| `symbol` | Mark shape, such as a circle or a diamond |
| `hover_name` | The bold title of the hover label |
| `animation_frame` | A play button and a slider with one frame per value |
| `labels` | Readable names for columns in axes, legends and hover labels |

This works when the data is **tidy**, with one row per observation and one column per variable.

### Which channel to use

Channels are not equally easy to read.
In perception experiments, people compare **positions on a shared axis** most accurately, then lengths, then angles and areas, and color the least precisely ([Cleveland & McGill, 1984](https://doi.org/10.1080/01621459.1984.10478080)).

**position > length > angle > area > color**

This is also why pie charts, which ask you to compare angles, are hard to read.
Examples 1 and 4 put their main comparison on a straight axis and use color only to say which group a mark belongs to.
Examples 2 and 3 trade some precision for something an axis cannot show, how a total splits and how a day cycles, and each says so.

In [ ]:
import pandas as pd
import plotly.express as px

# graph_objects is the lower-level part of Plotly, used for charts Plotly Express does not have
import plotly.graph_objects as go

# white background with light gridlines for every chart
px.defaults.template = "plotly_white"

<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

## Data for this notebook

**How does the Bay Area use bike share?**
[Bay Wheels](https://lyftbikes.com/system-data) is the Bay Area's bike share system.
It publishes a record of every trip, with its start and end time, start and end station, and rider type.
A **member** pays for an annual or monthly membership, and a **casual** rider pays per ride or per day.

**Where the data comes from**

- **Bike trips.** Bay Wheels publishes every trip as monthly CSV files on its [system data page](https://lyftbikes.com/system-data).
  We used all twelve files for 2025, about 4.35 million trips after removing trips shorter than a minute or longer than a day.
  That is too many rows to plot one by one, so for this notebook the trips were counted into four small tables in `data/raw/`, one for each example.
- **Weather.** Example 4 also uses daily rainfall at San Francisco International Airport, from [Meteostat](https://meteostat.net/en/station/72494), which compiles NOAA weather records (CC BY-NC 4.0 license).
  It is in `data/raw/sf_weather_2025.csv`.

All four charts share a few settings, so we define them once:

- **Blue and orange for every comparison of two groups**, with gray for anything neutral.
  This pair from the [Okabe-Ito palette](https://jfly.uni-koeln.de/color/) stays distinguishable for the most common types of color blindness ([Wilke, Ch. 19](https://clauswilke.com/dataviz/color-pitfalls.html)).
  Each chart's legend says which group is which.
  Example 4 draws the typical week in gray.
- **Readable labels**, passed to the `labels` argument so that `avg_trips` shows up as "Average trips" in axes, legends and hover labels.

In [ ]:
# Okabe-Ito blue and orange for every two-group comparison
day_type_colors = {
    "Weekday": "#0072B2",
    "Weekend": "#E69F00"
    }

rider_colors = {
    "Member": "#0072B2",
    "Casual": "#E69F00"
    }

# gray for the typical week, blue for the week being shown
week_colors = {
    "Typical week": "#BDBDBD",
    "This week": "#0072B2"
    }

# days in calendar order, for the animation in Example 3
day_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]

# readable names for the columns
column_labels = {
    "station": "Station",
    "day_type": "Day type",
    "avg_trips": "Average trips",
    "hour": "Hour of the day",
    "rider_type": "Rider type",
    "day_of_week": "Day",
    "week_of": "Week of",
    "hour_of_week": "Hour of the week",
    "trips": "Trips that hour",
    "date": "Date",
    "precip_mm": "Rain that day (mm)",
    "line": ""
    }

<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

## Example 1 - Comparing values with a dot plot

**Question.** At the 20 busiest stations, how does the number of trips on an average weekday compare with an average weekend day?

We need two values per station.
Grouped bars would draw two bars per station, and the eye would still compare only where each bar ends.
A **dot plot** keeps just those ends, as positions on a shared axis, the channel people read most accurately ([Wilke, Ch. 6](https://clauswilke.com/dataviz/visualizing-amounts.html)).
Sorting the stations by value turns the chart into a ranking.

The table has one column for weekdays and one for weekends.
Plotly Express needs one column for the value and one for the day type, so we reshape it with `.melt()`, which stacks the two columns into one.

In [ ]:
#Load the station table, already sorted with the busiest station first
stations = pd.read_csv("data/raw/bike_station_daytype_2025.csv")
stations.head()

In [ ]:
# keep the 20 busiest stations, with short column names for the two day types
top_20 = stations.head(20).rename(
    columns={"weekday_avg_trips": "Weekday", "weekend_avg_trips": "Weekend"}
    )

# stack the two day-type columns into one, so each row is one station on one day type
top_20_long = top_20.melt(
    id_vars="station",
    value_vars=["Weekday", "Weekend"],
    var_name="day_type",
    value_name="avg_trips"
    )

# station names sorted by weekday trips, busiest first, used to order the y-axis
station_order = top_20.sort_values("Weekday", ascending=False)["station"].tolist()

In [ ]:
fig = px.scatter(
    top_20_long,
    x="avg_trips",
    y="station",
    color="day_type",
    symbol="day_type",
    hover_name="station",
    category_orders={"station": station_order},
    color_discrete_map=day_type_colors,
    symbol_map={"Weekday": "circle", "Weekend": "diamond"},
    labels=column_labels,
    title="Transit-hub stations are busiest on weekdays, Golden Gate Park on weekends<br><sup>Average trips started per day at the 20 busiest Bay Wheels stations, 2025</sup>",
    height=650
    )

# make the dots (markers) bigger
fig.update_traces(marker_size=11)
fig.show()

**What to notice**

- The San Francisco Caltrain station leads on weekdays, with about 160 trips a day, but drops to about 59 on weekends.
- Sansome St at Sutter St, in the Financial District, falls from about 89 trips on weekdays to about 16 on weekends.
- At the bottom, 8th Ave at JFK Dr, in Golden Gate Park, more than doubles on weekends, from about 52 to about 123.
- Day type is shown twice, by color and by shape.
  This **redundant coding** keeps the chart readable in grayscale and for color-blind readers ([Wilke, Ch. 20](https://clauswilke.com/dataviz/redundant-coding.html)).

### ✏️ Your Turn: the same data as grouped bars

Draw `top_20_long` as a **grouped bar chart** instead of a dot plot, then compare the two.
Which chart makes it easier to see which stations are busier on weekends than on weekdays?

<details>
<summary>Hint</summary>

Copy the `px.scatter` cell above, change `px.scatter` to `px.bar`, delete the `symbol` and `symbol_map` lines, and add `barmode="group"` so the two bars sit side by side instead of on top of each other.
</details>

<details>
<summary>Check your answer</summary>

Both charts show the same numbers.
In the dot plot, the gap between the circle and the diamond sits on one line and is easy to judge.
With bars, your eye has to compare the ends of two separate bars, and the long bars add a lot of ink without adding information.
</details>

In [ ]:
# ✏️ Your Turn: draw top_20_long as a grouped bar chart

<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

## Example 2 - Following trips through categories with a Sankey diagram

**Question.** How do 4.35 million trips split by rider type, then by bike type, then by trip length?

A **Sankey diagram** draws each category as a box and each group of trips as a ribbon flowing between boxes, with the width of the ribbon set by the number of trips.
It answers "how does this total split, step by step?", which neither a bar chart nor a line chart shows in one picture ([Kosara et al., 2006](https://doi.org/10.1109/TVCG.2006.76)).
Here each ribbon keeps its rider color all the way across, so you can follow members and casual riders through every step.

Ribbon widths are read less precisely than positions on an axis, so use a Sankey to see the big splits and hover over a ribbon for exact numbers.

Plotly Express has no Sankey function, so this example uses `plotly.graph_objects`, the lower-level part of Plotly that Plotly Express is built on ([Plotly docs](https://plotly.com/python/sankey-diagram/)).
We imported it at the top as `go`, the short name everyone uses, just as Plotly Express is imported as `px`.
`go.Figure()` creates an empty figure, and `go.Sankey()` is the chart that goes inside it.
Instead of naming columns, you pass the boxes and the ribbons as lists.

In [ ]:
#Load the flows table, one row per ribbon
trip_flows = pd.read_csv("data/raw/bike_trip_flows_2025.csv")
trip_flows

In [ ]:
# every box in the diagram, left to right
node_labels = ["Member", "Casual", "Classic bike", "E-bike",
               "Under 10 min", "10-20 min", "20-30 min", "30 min or more"]

# where each box sits, from 0 to 1, so trip lengths run shortest to longest top to bottom
node_x = [0.001, 0.001, 0.5, 0.5, 0.999, 0.999, 0.999, 0.999]
node_y = [0.33, 0.85, 0.85, 0.36, 0.22, 0.6, 0.8, 0.9]

# go.Sankey refers to each box by its position in node_labels
trip_flows["source_id"] = trip_flows["source"].map(node_labels.index)
trip_flows["target_id"] = trip_flows["target"].map(node_labels.index)

# each ribbon takes its rider color, made see-through so crossings stay readable
ribbon_colors = {
    "Member": "rgba(0, 114, 178, 0.45)",
    "Casual": "rgba(230, 159, 0, 0.5)"
    }
trip_flows["color"] = trip_flows["rider_type"].map(ribbon_colors)

**How to read it.** Read it from left to right, where every trip starts at its rider type, flows through the bike it used, and ends at its trip length.
The wider a ribbon, the more trips it carries, and its color says whether those trips were by members (blue) or casual riders (orange).

In [ ]:
fig = go.Figure(
    go.Sankey(
        arrangement="snap",
        node=dict(
            label=node_labels,
            x=node_x,
            y=node_y,
            color=["#0072B2", "#E69F00"] + ["#9A9A9A"] * 6,
            pad=15,
            thickness=18
            ),
        link=dict(
            source=trip_flows["source_id"],
            target=trip_flows["target_id"],
            value=trip_flows["trips"],
            color=trip_flows["color"]
            )
        )
    )

fig.update_layout(
    title="Casual riders take 1 in 5 trips, but most trips of 30+ minutes<br><sup>Nearly 6 in 10 trips of 30 minutes or more were by casual riders (Bay Wheels, 2025)</sup>",
    height=600,
    margin=dict(t=90, b=40)
    )
fig.show()

**What to notice**

- Both rider types flow mostly into the E-bike box.
  About 78% of casual trips and 79% of member trips are on e-bikes.
- Follow the blue member ribbons into the trip-length boxes.
  Most member trips end in under 10 minutes, which fits short, regular trips such as commutes.
- Now follow the orange casual ribbons.
  They are a small share of all trips, but they make up most of the "30 min or more" box.
  Hover over it to read the numbers.

### ✏️ Your Turn: read the table behind the Sankey

The Sankey draws `trip_flows`, so you can also answer questions with the table directly.
How many trips of **30 minutes or more** did **casual** riders take on each bike type?

<details>
<summary>Hint</summary>

Keep the rows where two conditions are both true, using `&` and parentheses around each condition:

```python
trip_flows[(trip_flows["target"] == ...) & (trip_flows["rider_type"] == ...)]
```
</details>

<details>
<summary>Check your answer</summary>

About 102,000 on e-bikes and about 37,000 on classic bikes, so roughly 3 in 4 of casual riders' long trips were on e-bikes.
</details>

In [ ]:
# ✏️ Your Turn: casual riders' trips of 30 minutes or more, by bike type

<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

## Example 3 - A 24-hour clock of riding, animated through the week

**Question.** How does the daily rhythm of riding change from Monday to Sunday, for members and for casual riders?

The hours of the day form a cycle, so 11 pm sits right next to midnight.
A **polar chart** wraps the hours around a circle like a clock face, so the cycle has no artificial break at midnight ([Wilke, Ch. 3](https://clauswilke.com/dataviz/coordinate-systems-axes.html)).
The distance from the center is the average number of trips started in that hour.
Distance from a center is read less precisely than position along a straight axis, so use this chart to see the **shape** of a day, and hover for exact numbers.

`animation_frame="day_of_week"` steps through the week one day at a time, and `range_r` fixes the radius, so every frame uses the same scale.
The play button normally stops after Sunday, so we give it the week repeated 50 times, and it keeps looping until you click pause.

In [ ]:
#Load the clock table, average trips started in each hour of each day of the week
clock = pd.read_csv("data/raw/bike_clock_2025.csv")

# 24 hours around a 360-degree circle, 15 degrees per hour
clock["angle"] = clock["hour"] * 15

clock.head()

In [ ]:
fig = px.line_polar(
    clock,
    r="avg_trips",
    theta="angle",
    color="rider_type",
    line_close=True,
    hover_data={"hour": True, "angle": False},
    animation_frame="day_of_week",
    range_r=[0, 1400],
    category_orders={"day_of_week": day_order, "rider_type": ["Member", "Casual"]},
    color_discrete_map=rider_colors,
    labels=column_labels,
    title="Weekdays have two commute peaks, weekends one long afternoon<br><sup>Average Bay Wheels trips started in each hour, 2025. Press play to step through the week.</sup>",
    height=650
    )

fig.update_traces(line_width=3)

# turn the circle into a clock face, with midnight at the top
fig.update_polars(
    angularaxis_direction="clockwise",
    angularaxis_rotation=90,
    angularaxis_tickvals=[0, 45, 90, 135, 180, 225, 270, 315],
    angularaxis_ticktext=["midnight", "3 am", "6 am", "9 am", "noon", "3 pm", "6 pm", "9 pm"]
    )

# play the week over and over until the pause button is clicked, one second per day
play_button = fig.layout.updatemenus[0].buttons[0]
play_settings = play_button.args[1]
play_settings["frame"]["duration"] = 1000
play_button.args = [day_order * 50, play_settings]

fig.show()

**What to notice**

- Monday to Friday, the blue member outline has two sharp spikes, around 8 am and 5 pm.
  That is the commute.
- On Saturday and Sunday the spikes disappear, and both outlines swell into one long bulge from late morning into the evening.
- The orange casual outline grows on weekends, peaking at about 370 trips an hour on Saturday against about 280 on a typical weekday.
- Compare Friday with Tuesday.
  Friday's commute spikes are the smallest of the week, about 990 member trips at the peak against about 1,350 on Tuesday.
  What might explain that?

### ✏️ Your Turn: the clock as an ordinary line chart

Draw **Monday only** as an ordinary line chart, with the hour on the x-axis and one line per rider type.
Which chart makes it easier to read the exact hour of the evening peak?
Which one shows better that the day wraps around from 11 pm to midnight?

<details>
<summary>Hint</summary>

First keep only Monday's rows with `clock[clock["day_of_week"] == "Monday"]`.
Then use `px.line` with `x="hour"`, `y="avg_trips"` and `color="rider_type"`, and reuse `color_discrete_map=rider_colors` and `labels=column_labels`.
</details>

<details>
<summary>Check your answer</summary>

The line chart makes the peak easy to read, at 5 pm (hour 17) with about 1,130 member trips.
The clock shows that 11 pm sits right next to midnight, which the line chart splits across its two ends.
Each chart is better at one job.
</details>

In [ ]:
# ✏️ Your Turn: Monday as a line chart

<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

## Example 4 - The weekly heartbeat, animated through the year

**Question.** Which weeks of 2025 broke the normal rhythm of riding, and why?

Example 3 averaged the whole year, which hides the unusual days.
Here each frame is one real week, hour by hour from Monday to Sunday, drawn over the **typical week**, the median of all 51 full weeks of 2025, as a wide gray line.

The gray line stays on screen in every frame.
That matters, because research on animated line charts finds that people compare better when they do not have to remember earlier frames, such as when the earlier path stays visible ([Hu et al., 2024](https://arxiv.org/abs/2408.04799)).
`range_y` fixes the y-axis for the same reason ([Plotly docs](https://plotly.com/python/animations/)).

We also join each hour to that day's rainfall at San Francisco International Airport with `.merge()`, so hovering over the line shows how much rain fell.

In [ ]:
#Load trips per hour for every full week, and daily weather
weeks = pd.read_csv("data/raw/bike_weekly_hours_2025.csv")
weather = pd.read_csv("data/raw/sf_weather_2025.csv")

# add each day's rainfall to every hour of that day
weeks = weeks.merge(
    weather[["date", "precip_mm"]],
    on="date"
    )

# the typical week, the median trips in each hour of the week across all 51 weeks
typical = weeks.groupby("hour_of_week", as_index=False)["trips"].median()

# give every week its own copy of the typical week, so both lines appear in every frame
typical_rows = weeks[["week_of", "hour_of_week"]].merge(
    typical,
    on="hour_of_week"
    )
typical_rows["line"] = "Typical week"
weeks["line"] = "This week"

heartbeat = pd.concat([typical_rows, weeks])

In [ ]:
fig = px.line(
    heartbeat,
    x="hour_of_week",
    y="trips",
    color="line",
    hover_data=["date", "precip_mm"],
    animation_frame="week_of",
    range_y=[0, 2400],
    category_orders={"line": ["Typical week", "This week"]},
    color_discrete_map=week_colors,
    labels=column_labels,
    title="Storms and holidays break the weekly rhythm<br><sup>Bay Wheels trips started each hour, one week per frame, 2025. Press play.</sup>",
    height=500
    )

# a wide gray typical week behind a thinner line for this week
fig.update_traces(line_width=9, selector=dict(name="Typical week"))
fig.update_traces(line_width=2.5, selector=dict(name="This week"))

# label the x-axis with days instead of hour numbers
fig.update_xaxes(
    tickvals=[12, 36, 60, 84, 108, 132, 156],
    ticktext=["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
    )

# slow the animation down (milliseconds per frame and per transition)
fig.layout.updatemenus[0].buttons[0].args[1]["frame"]["duration"] = 700
fig.layout.updatemenus[0].buttons[0].args[1]["transition"]["duration"] = 300

fig.show()

**What to notice**

Drag the slider to these weeks, and hover over the dips to read the rainfall.

- **Week of Feb 3.** Tuesday and Thursday collapse to about a third of a typical day.
  February 4 and 6 brought 44.5 mm and 17 mm of rain.
- **Week of Feb 10.** Thursday, February 13, had 48 mm of rain and about a third of the usual trips.
- **Week of Nov 24.** Thanksgiving Thursday and the Friday after lose their commute peaks and drop to about a third and a half of a typical day.
- **Week of Dec 22.** The whole week runs at about a third of normal, with Christmas and three days of heavy rain (December 22, 24 and 25).
- **Weeks in September and October** run about 30% above the typical week.
  Part of that is fall weather and part is the stations Bay Wheels added during the year.

### Animation or small multiples?

Animation is a great way to tell a story to an audience, but it is weaker for analysis, because comparing frames means remembering what is no longer on screen.
In a controlled study, people answered questions about trends more accurately with static small multiples than with an animated chart, even though they enjoyed the animation more ([Robertson et al., 2008](https://doi.org/10.1109/TVCG.2008.125)).
The typical-week line is this example's way of keeping a fixed comparison on screen.
To analyse the year, you would still want a static chart, such as one line per week in small multiples.

<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

## Summary - Plotly Express Walkthrough

| Example | Question | Chart | Key arguments |
|---|---|---|---|
| 1 | Compare two values across many items | Dot plot | `px.scatter`, `symbol`, `category_orders` |
| 2 | Show how a total splits, step by step | Sankey diagram | `go.Sankey` with `node` and `link` |
| 3 | Show how a daily cycle changes through the week | Animated polar "clock" | `px.line_polar`, `animation_frame`, `range_r` |
| 4 | Find the weeks that broke the pattern | Animated line chart over a fixed reference | `merge`, `px.line`, `animation_frame`, `range_y` |

## Key Takeaways

1. **Every chart is a set of choices about columns** - the 20 busiest stations became dots once `x="avg_trips"` set their position and `color="day_type"` split weekdays from weekends.
2. **Put the comparison you care about on an axis** - the Caltrain station's weekend drop jumps out of the dot plot because the eye reads position more accurately than length, angle, area or color.
3. **When Plotly Express runs out, reach for `go`** - the Sankey needed `go.Sankey`, which swaps column names for lists of boxes and ribbons and in return shows one total splitting three ways in a single picture.
4. **Give each group a color and keep it** - members stayed blue and casual riders orange in Examples 2 and 3, so readers learned the code once and could read the clock at a glance.
5. **Animate to tell the story, not to find it** - the February storms leap out of the heartbeat only because the typical week stays on screen and the axes never move.

To share any of these charts, run `fig.write_html("my_chart.html")` and open the file in a browser.
The chart stays fully interactive.

## Further reading

- [Plotly Express overview](https://plotly.com/python/plotly-express/) and [Plotly animations](https://plotly.com/python/animations/)
- Claus O. Wilke, [*Fundamentals of Data Visualization*](https://clauswilke.com/dataviz/), free online
- [From Data to Viz](https://www.data-to-viz.com/), a decision tree for picking a chart type
- [Plotly Sankey diagrams](https://plotly.com/python/sankey-diagram/)
- [Bay Wheels system data](https://lyftbikes.com/system-data), the source of the trip records
- [Meteostat](https://meteostat.net/), the source of the daily weather records